In [ ]:
!pip install ucimlrepo


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install ucimlrepo

https://archive.ics.uci.edu/dataset/222/bank+marketing

In [2]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split

In [3]:
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (45211, 16)
y shape: (45211, 1)


In [7]:
display(X.head())

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [8]:
display(y.head())

,y
0,no
1,no
2,no
3,no
4,no


In [9]:
print("Feature data types:")
print(X.dtypes)

print("\nTarget data type:")
print(y.dtypes)

Feature data types:
age             int64
job            object
marital        object
education      object
default        object
balance         int64
housing        object
loan           object
contact        object
day_of_week     int64
month          object
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome       object
dtype: object

Target data type:
y    object
dtype: object


In [10]:
print(X.isna().sum())
print(y.isna().sum())

age                0
job              288
marital            0
education       1857
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
dtype: int64
y    0
dtype: int64


In [11]:
# SPLIT: 86% TRAIN, 7% VALIDATION, 7% TEST

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.14,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (38881, 16)
X_val: (3165, 16)
X_test: (3165, 16)
y_train: (38881, 1)
y_val: (3165, 1)
y_test: (3165, 1)


In [12]:
cols_to_drop = ["duration"]

X_train_clean = X_train.drop(columns=cols_to_drop)
X_val_clean = X_val.drop(columns=cols_to_drop)

print("X_train_clean:", X_train_clean.shape)
print("X_val_clean:", X_val_clean.shape)

X_train_clean: (38881, 15)
X_val_clean: (3165, 15)


In [13]:
numeric_features = X_train_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train_clean.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous']

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [14]:
# numeric: median imputation
# categorical: Unknown imputation + OneHotEncoding

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created.")

Preprocessor created.


In [15]:
# Baseline model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

baseline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        criterion="gini",
        random_state=42,
        n_jobs=-1
    ))
])

baseline_rf.fit(X_train_clean, y_train)

val_pred = baseline_rf.predict(X_val_clean)

print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print("Validation Macro F1:", f1_score(y_val, val_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_val, val_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

c:\Users\zirad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Validation Accuracy: 0.8875197472353871
Validation Macro F1: 0.617658157060331

Classification Report:
              precision    recall  f1-score   support

          no       0.90      0.98      0.94      2794
         yes       0.56      0.20      0.30       371

    accuracy                           0.89      3165
   macro avg       0.73      0.59      0.62      3165
weighted avg       0.86      0.89      0.86      3165


Confusion Matrix:
[[2734   60]
 [ 296   75]]


2734 = correctly predicted no
60   = predicted yes but actually no
296  = predicted no but actually yes
75   = correctly predicted yes

In [16]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [200, 300, 500, 800],
    "model__criterion": ["gini", "entropy", "log_loss"],
    "model__max_depth": [None, 5, 10, 15, 20, 30],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__max_features": ["sqrt", "log2", None],
    "model__bootstrap": [True, False],
    "model__class_weight": [None, "balanced", "balanced_subsample"]
}

search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_grid,
    n_iter=50,
    scoring="f1",          # positive class = "yes"
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

search.fit(X_train_clean, y_train)
best_rf = search.best_estimator_
print(search.best_params_)
print(search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


c:\Users\zirad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(
c:\Users\zirad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the train scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(
c:\Users\zirad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 

{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 8, 'model__max_features': 'log2', 'model__max_depth': 10, 'model__criterion': 'entropy', 'model__class_weight': 'balanced_subsample', 'model__bootstrap': False}
nan


In [17]:
val_pred_tuned = best_rf.predict(X_val_clean)

print("Validation Accuracy:", accuracy_score(y_val, val_pred_tuned))
print("Validation Macro F1:", f1_score(y_val, val_pred_tuned, average="macro"))
print("Validation F1 for yes:", f1_score(y_val, val_pred_tuned, pos_label="yes"))

print("\nClassification Report:")
print(classification_report(y_val, val_pred_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_tuned))

Validation Accuracy: 0.8060031595576619
Validation Macro F1: 0.6547190931381259
Validation F1 for yes: 0.4261682242990654

Classification Report:
              precision    recall  f1-score   support

          no       0.94      0.83      0.88      2794
         yes       0.33      0.61      0.43       371

    accuracy                           0.81      3165
   macro avg       0.63      0.72      0.65      3165
weighted avg       0.87      0.81      0.83      3165


Confusion Matrix:
[[2323  471]
 [ 143  228]]


In [18]:
# ============================================================
# 18. TOP 10 MODELS FROM RANDOMIZED SEARCH
# ============================================================

results_df = pd.DataFrame(search.cv_results_)

top_cols = [
    "mean_test_score",
    "mean_train_score",
    "param_model__n_estimators",
    "param_model__criterion",
    "param_model__max_depth",
    "param_model__min_samples_split",
    "param_model__min_samples_leaf",
    "param_model__max_features",
    "param_model__bootstrap",
    "param_model__class_weight"
]

top_results = (
    results_df[top_cols]
    .sort_values("mean_test_score", ascending=False)
    .head(10)
)

print(top_results)

   mean_test_score  mean_train_score  param_model__n_estimators  \
0              NaN               NaN                        500   
1              NaN               NaN                        800   
2              NaN               NaN                        200   
3              NaN               NaN                        500   
4              NaN               NaN                        800   
5              NaN               NaN                        200   
6              NaN               NaN                        200   
7              NaN               NaN                        500   
8              NaN               NaN                        300   
9              NaN               NaN                        500   

  param_model__criterion param_model__max_depth  \
0                entropy                     10   
1                entropy                     30   
2                   gini                     15   
3                   gini                     20   
4      